# tm_final_33 - Financial Tweet Sentiment Classification
## Text Mining 2025/2026 - NOVA IMS
### Ensemble Pipeline — Restart & Run All

**Primary model:** `nickmuchi/finbert-tone-finetuned-fintwitter-classification` (FinBERT pre-trained on twitter-financial-news-sentiment — the exact same domain).  
Fine-tuned with 5-fold stratified CV, 10 epochs, cosine LR schedule, label smoothing, fp16 + GradScaler, best-checkpoint-per-fold.  
**Text fix applied:** `ftfy` mojibake correction + truncation/URL cleanup (affects 23.5% of tweets; +0.61pp OOF F1-macro vs raw text).

**Ensemble:** Soft-vote of multiple complementary models (FinBERT variants, RoBERTa-large, DeBERTa-v3) whose pre-computed test probabilities are stored in `results/predictions/prob_test_*.csv`. If those files are present the ensemble is used for `pred_33.csv`; otherwise falls back to the primary model alone.

| Configuration | OOF F1-macro |
|---|---|
| RoBERTa-large baseline (previous) | 0.8873 |
| FinBERT fix-text 10ep (primary model) | **0.9075** |
| Ensemble (7+ models) | **≥ 0.9141** |

**Runtime:** ~25 min on CUDA GPU (RTX 5070, batch 16, fp16). **Instructions:** Kernel → Restart Kernel and Run All Cells.

In [ ]:
# Cell 1: Imports and reproducibility
import os, sys, re, time, gc, warnings, tempfile
warnings.filterwarnings('ignore')
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import ftfy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (f1_score, accuracy_score, precision_score,
                             recall_score, classification_report)
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          get_cosine_schedule_with_warmup)

SEED = 42

def seed_all(seed=SEED):
    import random
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_all()
print(f'Seed fixed: {SEED}')
print(f'Python: {sys.version[:20]} | torch {torch.__version__}')
print(f'ftfy {ftfy.__version__}')

In [ ]:
# Cell 3: Preprocessing — ftfy mojibake fix + truncation cleanup
# The dataset (zeroshot/twitter-financial-news-sentiment) has 23.5% of tweets with
# corrupted UTF-8 characters (e.g. â€™ instead of ') and 16.4% truncated with
# U+FFFD replacement chars. ftfy detects and fixes both automatically.
# Ablation: fix_text +0.61pp OOF F1-macro vs raw text (90.75% vs 90.14%).

_TRUNC_RE = re.compile(r'[�…°�…]+\s*(https?://\S*)?$')
_URL_RE    = re.compile(r'https?://\S+')
_TRAIL_RE  = re.compile(r'[\s\-–:]+$')

def fix_tweet(text: str) -> str:
    """Fix mojibake (ftfy) and remove truncation artefacts + bare URLs."""
    text = ftfy.fix_text(str(text))
    text = _TRUNC_RE.sub('', text)
    text = _URL_RE.sub('', text)
    return _TRAIL_RE.sub('', text).strip()

train = pd.read_csv('data/raw/train.csv')
test  = pd.read_csv('data/raw/test.csv')

texts      = [fix_tweet(t) for t in train['text'].tolist()]
test_texts = [fix_tweet(t) for t in test['text'].tolist()]
y = train['label'].values

print(f'Train: {train.shape} | Test: {test.shape}')
print('Label distribution (0=Bearish, 1=Bullish, 2=Neutral):')
print(train['label'].value_counts().sort_index())
print(f'\nSample before: {train["text"].iloc[1][:90]}')
print(f'Sample after:  {texts[1][:90]}')

In [ ]:
# Cell 4: Configuration and device selection
MODEL_NAME   = 'nickmuchi/finbert-tone-finetuned-fintwitter-classification'
MAXLEN       = 128
LR           = 5e-6
WEIGHT_DECAY = 0.01
EPOCHS       = 10
N_FOLDS      = 5
WARMUP_RATIO = 0.06
LABEL_SMOOTH = 0.05
GRAD_ACCUM   = 1

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    BATCH, EVAL_BATCH = 16, 32
    AMP_DTYPE = torch.float16
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    USE_SCALER = True
    print(f'GPU: {torch.cuda.get_device_name(0)} | '
          f'VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB | '
          f'fp16 + GradScaler')
else:
    BATCH, EVAL_BATCH = 8, 32
    AMP_DTYPE, USE_SCALER = None, False
    torch.set_num_threads(min(12, os.cpu_count()))
    print('CPU fallback')

# Inverse-frequency class weights
counts = np.bincount(y, minlength=3)
class_weights = torch.tensor(len(y) / (3 * counts), dtype=torch.float32)
print(f'class weights: {[round(x,3) for x in class_weights.tolist()]}')

tok = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
# Cell 5: Model builder, training loop and inference helpers
# Improvements vs notebook v1: GradScaler (fp16), cosine LR warmup,
# label smoothing, best-checkpoint-per-fold (val F1, not last epoch).

def build_model():
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=3, ignore_mismatched_sizes=True,
        dtype=torch.float32,   # fp32 params required for GradScaler
    ).to(DEVICE)


@torch.no_grad()
def predict_proba(model, txts):
    model.eval()
    out = []
    for i in range(0, len(txts), EVAL_BATCH):
        enc = tok(txts[i:i+EVAL_BATCH], padding=True, truncation=True,
                  max_length=MAXLEN, return_tensors='pt')
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        if DEVICE == 'cuda' and AMP_DTYPE is not None:
            with torch.autocast(device_type='cuda', dtype=AMP_DTYPE):
                logits = model(**enc).logits
        else:
            logits = model(**enc).logits
        out.append(torch.softmax(logits.float(), dim=1).cpu().numpy())
    return np.vstack(out)


def run_fold(tr_texts, tr_labels, va_texts, va_y, all_test_texts, fold):
    """Full fold: train with best-ckpt, return (best_va_proba, best_test_proba)."""
    seed_all(SEED + fold)
    model = build_model()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    steps_per_epoch = int(np.ceil(len(tr_texts) / BATCH))
    total_steps = (steps_per_epoch // GRAD_ACCUM) * EPOCHS
    warmup_steps = int(WARMUP_RATIO * total_steps)
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE),
                                  label_smoothing=LABEL_SMOOTH)
    scaler = torch.amp.GradScaler('cuda') if USE_SCALER else None

    best_f1, best_va_proba, best_ckpt_path = -1.0, None, None
    n = len(tr_texts)

    with tempfile.TemporaryDirectory() as tmpdir:
        ckpt = Path(tmpdir) / 'best.pt'
        for epoch in range(EPOCHS):
            model.train()
            order = np.random.permutation(n)
            running, t0 = 0.0, time.time()
            optimizer.zero_grad(set_to_none=True)
            for step, i in enumerate(range(0, n, BATCH)):
                bidx = order[i:i+BATCH]
                bt = [tr_texts[j] for j in bidx]
                bl = torch.tensor([tr_labels[j] for j in bidx], dtype=torch.long, device=DEVICE)
                enc = tok(bt, padding=True, truncation=True, max_length=MAXLEN, return_tensors='pt')
                enc = {k: v.to(DEVICE) for k, v in enc.items()}
                if DEVICE == 'cuda' and AMP_DTYPE is not None:
                    with torch.autocast(device_type='cuda', dtype=AMP_DTYPE):
                        loss = loss_fn(model(**enc).logits, bl)
                else:
                    loss = loss_fn(model(**enc).logits, bl)
                loss = loss / GRAD_ACCUM
                if scaler: scaler.scale(loss).backward()
                else:       loss.backward()
                running += float(loss.item()) * GRAD_ACCUM
                if (step+1) % GRAD_ACCUM == 0 or (i+BATCH) >= n:
                    if scaler: scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    if scaler: scaler.step(optimizer); scaler.update()
                    else:      optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad(set_to_none=True)
            # validate
            va_proba = predict_proba(model, va_texts)
            ep_f1 = f1_score(va_y, va_proba.argmax(1), average='macro')
            star = ' *** best ***' if ep_f1 > best_f1 else ''
            print(f'    fold {fold+1} epoch {epoch+1}/{EPOCHS} '
                  f'loss={running/steps_per_epoch:.4f} val_f1={ep_f1:.6f} '
                  f'({time.time()-t0:.0f}s){star}')
            if ep_f1 > best_f1:
                best_f1 = ep_f1
                best_va_proba = va_proba.copy()
                torch.save(model.state_dict(), ckpt)
        # reload best checkpoint for test predictions
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
        best_test_proba = predict_proba(model, all_test_texts)

    print(f'  fold {fold+1} best val macro-F1={best_f1:.6f}')
    return best_va_proba, best_test_proba, best_f1

print('Helpers defined.')

In [ ]:
# Cell 6: 5-fold CV fine-tuning -> OOF F1-macro + averaged test probabilities
cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_proba      = np.zeros((len(y), 3), dtype=np.float32)
test_proba_sum = np.zeros((len(test_texts), 3), dtype=np.float32)
fold_f1 = []
t_start = time.time()

for fold, (tr_idx, va_idx) in enumerate(cv.split(texts, y)):
    print(f'\n=== Fold {fold+1}/{N_FOLDS} ===')
    tr_texts_fold = [texts[i] for i in tr_idx]
    tr_labels     = [int(y[i]) for i in tr_idx]
    va_texts_fold = [texts[i] for i in va_idx]
    va_y          = y[va_idx]

    va_proba, test_proba_fold, best_f1 = run_fold(
        tr_texts_fold, tr_labels, va_texts_fold, va_y, test_texts, fold)

    oof_proba[va_idx] = va_proba
    test_proba_sum   += test_proba_fold
    fold_f1.append(best_f1)

    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

test_proba_main = test_proba_sum / N_FOLDS
print(f'\nDone in {(time.time()-t_start)/60:.1f} min.')

In [ ]:
# Cell 7: Out-of-fold performance (honest, leak-free estimate)
oof_pred = oof_proba.argmax(1)
oof_f1   = f1_score(y, oof_pred, average='macro')

print(f'OOF F1-macro : {oof_f1:.4f}  (per-fold {[round(x,4) for x in fold_f1]}, '
      f'std {np.std(fold_f1):.4f})')
print(f'Accuracy     : {accuracy_score(y, oof_pred):.4f}')
print(f'Precision-mac: {precision_score(y, oof_pred, average="macro"):.4f}')
print(f'Recall-macro : {recall_score(y, oof_pred, average="macro"):.4f}')
print()
print(classification_report(y, oof_pred, target_names=['Bearish', 'Bullish', 'Neutral']))

In [ ]:
# Cell 7b: Ensemble — load pre-computed test probabilities from other models
# Pre-computed prob_test_*.csv files (committed to repo) contain soft probabilities
# from: FinBERT 10ep raw, FinBERT 7ep, FinBERT 5ep, RoBERTa-large, DeBERTa-v3-large,
# DeBERTa-v3-base-finance, and any new experiments in results/predictions/.
# Weights: each pre-computed model = 1.0 vote, deberta_base = 0.5 vote.

PRED_DIR = Path('results/predictions')
LIGHT_WEIGHT_TAGS = {'deberta_base_finance_fixtext'}  # weaker models get 0.5 weight

ensemble_sum   = test_proba_main.copy()   # start with this notebook's primary model
ensemble_count = 1.0

csv_probas = sorted(PRED_DIR.glob('prob_test_*.csv'))
loaded_tags = []
for csv_path in csv_probas:
    tag = csv_path.stem.replace('prob_test_', '')
    # skip if it's the same model we already trained above
    if 'finbert_fintwitter_10ep_fixtext' in tag:
        continue
    try:
        proba = pd.read_csv(csv_path)[['p0','p1','p2']].values.astype(np.float32)
        if proba.shape == (len(test_texts), 3):
            w = 0.5 if tag in LIGHT_WEIGHT_TAGS else 1.0
            ensemble_sum   += proba * w
            ensemble_count += w
            loaded_tags.append(f'{tag} (w={w})')
    except Exception as e:
        print(f'  Skipping {csv_path.name}: {e}')

test_proba_final = ensemble_sum / ensemble_count

if loaded_tags:
    ens_f1 = f1_score(y, (ensemble_sum / ensemble_count).argmax(1), average='macro')
    print(f'Ensemble loaded {len(loaded_tags)} extra models:')
    for t in loaded_tags:
        print(f'  + {t}')
    print(f'\nPrimary model OOF F1-macro : {oof_f1:.4f}')
    print(f'Ensemble test distribution  : using {ensemble_count:.1f} effective votes')
else:
    test_proba_final = test_proba_main
    print('No pre-computed model CSVs found — using primary model only.')
    print(f'Primary model OOF F1-macro: {oof_f1:.4f}')

In [ ]:
# Cell 8: Generate Predictions and Save pred_33.csv
test_pred  = test_proba_final.argmax(1)
submission = pd.DataFrame({'id': test['id'], 'label': test_pred})
submission.to_csv('pred_33.csv', index=False)
os.makedirs('results/predictions', exist_ok=True)
submission.to_csv('results/predictions/pred_final.csv', index=False)

print(f'Predictions saved: pred_33.csv ({len(submission)} rows)')
print('Distribution:')
print(submission['label'].value_counts().sort_index()
      .rename({0: 'Bearish', 1: 'Bullish', 2: 'Neutral'}))
print()

assert len(submission) == len(test), f'Expected {len(test)}, got {len(submission)}'
assert submission['label'].nunique() == 3, 'Model predicts fewer than 3 classes!'
assert list(submission.columns) == ['id', 'label'], 'Columns must be exactly [id, label]'
print('All assertions PASSED.')
print()
print(submission.head(10).to_string(index=False))